In [ ]:
# Build output DataFrame — one row per series per month
records = []

for actual, forecast, original_series in zip(val_inv, pred_series_inv, val_list):

    # Get the series label from unscaled val_list (retains original string value)
    series_name = original_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    actual_values   = actual.values().flatten()
    forecast_values = forecast.values().flatten()

    for month, act, pred in zip(months, actual_values, forecast_values):
        records.append({
            'MONTH_OF_SALE'                  : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'ACTUAL_SALES'                   : round(float(act),  2),
            'PREDICTED_SALES'                : round(float(pred), 2)
        })

df_output = pd.DataFrame(records)
df_output['MONTH_OF_SALE'] = pd.to_datetime(df_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_output = df_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_output.shape}')
print(f'Months       : {df_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_output.head(10)

In [ ]:
['DEALER_CODE',
 'CAL_DATE',
 'CUSTOM_DAY',
 'ACCNT_TYPE_CD',
 'X_CUSTOMER_TYPE',
 'SAP_CODE',
 'ACCOUNT_NAME',
 'ORDER_WID',
 'ROW_WID',
 'X_SUPPLIER_WID',
 'ORDER_TYPE',
 'PROD_WID',
 'X_CITY_CATEGORY',
 'X_FINANCE_FLG',
 'FINANCIER_NAME',
 'X_EXCHANGE_FLG',
 'MODEL',
 'SKU',
 'HMCL_BILLING_TYPE',
 'INVOICE_AMOUNT',
 'CGST',
 'SGST',
 'UTGST',
 'IGST',
 'CESS',
 'INVOICED_SALES',
 'CANCELLED_SALES',
 'RETURNED_SALES',
 'ORDER_NUM',
 'UPGRADE_STATUS',
 'NET_SALES']

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import col, to_date, months_between, lit
from snowflake.snowpark.types import IntegerType

# ── Step 1: Load tables as Snowpark DataFrames ────────────────────────────────
ecr_sf      = session.table("MOP_DATABASE.SOQ.ECR_SALES")
mapping_sf  = session.table("MOP_DATABASE.SOQ.SKU_MODEL_FAMILY_MAPPING")

# ── Step 2: Push forecast_df into Snowflake as a temp table ──────────────────
forecast_snow = session.create_dataframe(forecast_df)
forecast_snow = (
    forecast_snow
    .with_column('DEALER_CODE',       F.split(col('PARENT_DEALER_CODE_MODEL_FAMILY'), F.lit('<>'))[0])
    .with_column('MODEL_FAMILY_CODE', F.split(col('PARENT_DEALER_CODE_MODEL_FAMILY'), F.lit('<>'))[1])
    .with_column_renamed('NET_SALES', 'PREDICTED_SALES_FAMILY')
    .with_column('MONTH_OF_SALE', F.to_date(col('MONTH_OF_SALE')))
)

# ── Step 3: Prepare ecr_sales with CAL_DATE as date ──────────────────────────
ecr_sf = ecr_sf.with_column('CAL_DATE', F.to_date(col('CAL_DATE')))

# ── Step 4: Cross join forecast months to ecr_sales, filter rolling 3M window ─
# This replaces the Python loop entirely — Snowflake does the filtering per month
forecast_months = forecast_snow.select('MONTH_OF_SALE').distinct()

ecr_with_month = ecr_sf.join(forecast_months, how='cross')

ecr_windowed = ecr_with_month.filter(
    (col('CAL_DATE') >= F.add_months(col('MONTH_OF_SALE'), F.lit(-3))) &
    (col('CAL_DATE') <  col('MONTH_OF_SALE'))   # strictly before forecast month
)

# ── Step 5: Aggregate to dealer + SKU level per forecast month ────────────────
sku_history = (
    ecr_windowed
    .group_by(['MONTH_OF_SALE', 'DEALER_CODE', 'SKU'])
    .agg(F.sum('NET_SALES').alias('SKU_SALES_3M'))
)

# ── Step 6: Bring in MODEL_FAMILY_CODE ────────────────────────────────────────
sku_history = sku_history.join(
    mapping_sf.select('SKU', 'MODEL_FAMILY_CODE'),
    on='SKU',
    how='left'
)

# ── Step 7: Family totals per forecast month ──────────────────────────────────
family_totals = (
    sku_history
    .group_by(['MONTH_OF_SALE', 'DEALER_CODE', 'MODEL_FAMILY_CODE'])
    .agg(F.sum('SKU_SALES_3M').alias('FAMILY_SALES_3M'))
)

# ── Step 8: SKU weights ───────────────────────────────────────────────────────
sku_weights = sku_history.join(
    family_totals,
    on=['MONTH_OF_SALE', 'DEALER_CODE', 'MODEL_FAMILY_CODE'],
    how='left'
)

sku_weights = (
    sku_weights
    .with_column(
        'SKU_WEIGHT',
        F.when(col('FAMILY_SALES_3M') == 0, F.lit(None))
         .otherwise(col('SKU_SALES_3M') / col('FAMILY_SALES_3M'))
    )
    .with_column(
        'NO_HISTORY_FLAG',
        F.when(col('FAMILY_SALES_3M') == 0, F.lit(True)).otherwise(F.lit(False))
    )
)

# ── Step 9: Join weights to forecast ─────────────────────────────────────────
disaggregated = forecast_snow.join(
    sku_weights.select(
        'MONTH_OF_SALE', 'DEALER_CODE', 'MODEL_FAMILY_CODE',
        'SKU', 'SKU_WEIGHT', 'NO_HISTORY_FLAG'
    ),
    on=['MONTH_OF_SALE', 'DEALER_CODE', 'MODEL_FAMILY_CODE'],
    how='left'
)

# ── Step 10: Disaggregate ─────────────────────────────────────────────────────
disaggregated = disaggregated.with_column(
    'PREDICTED_SALES_SKU',
    F.when(col('NO_HISTORY_FLAG') == True, F.lit(None))
     .otherwise(F.round(col('PREDICTED_SALES_FAMILY') * col('SKU_WEIGHT'), 0))
)

# ── Step 11: Final output columns ─────────────────────────────────────────────
output = disaggregated.select(
    'MONTH_OF_SALE',
    'PARENT_DEALER_CODE_MODEL_FAMILY',
    'DEALER_CODE',
    'MODEL_FAMILY_CODE',
    'SKU',
    'SKU_WEIGHT',
    'PREDICTED_SALES_FAMILY',
    'PREDICTED_SALES_SKU',
    'NO_HISTORY_FLAG'
).sort('MONTH_OF_SALE', 'PARENT_DEALER_CODE_MODEL_FAMILY', 'SKU')

# ── Sanity check ──────────────────────────────────────────────────────────────
check = (
    output
    .filter(col('NO_HISTORY_FLAG') == False)
    .group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE_MODEL_FAMILY'])
    .agg(
        F.max('PREDICTED_SALES_FAMILY').alias('ORIGINAL_FORECAST'),
        F.sum('PREDICTED_SALES_SKU').alias('REAGGREGATED_SUM')
    )
    .with_column('DIFF', F.abs(col('ORIGINAL_FORECAST') - col('REAGGREGATED_SUM')))
)

print(f"Max reaggregation diff: {check.agg(F.max('DIFF')).collect()[0][0]}")
print(f"NO_HISTORY_FLAG rows  : {output.filter(col('NO_HISTORY_FLAG') == True).count()}")
print(f"Output row count      : {output.count()}")

output.to_pandas().head(10)